In [ ]:
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModel
from datasets import Dataset

class WordEmbeddingExtractor:
    def __init__(self, model_dir, dataset_path):
        """
        Initialize the embedding extractor with model directory and dataset.
        """
        self.model_dir = model_dir
        self.tokenizer = AutoTokenizer.from_pretrained(model_dir, model_max_length=512, truncation=True)
        self.model = AutoModel.from_pretrained(model_dir)
        self.model.eval()
        
        # Load dataset
        self.dataset = Dataset.from_file(dataset_path)
        print(f"Loaded dataset with {len(self.dataset)} entries.")

    def get_word_ids(self, word):
        """
        Get token IDs for a given word.
        """
        return self.tokenizer.encode(word, add_special_tokens=False)

    def filter_dataset(self, word):
        """
        Filter dataset examples containing the given word.
        """
        word_ids = self.get_word_ids(word)

        filtered_examples = [
            example for example in self.dataset
            if any(
                example["input_ids"][i:i + len(word_ids)] == word_ids
                for i in range(len(example["input_ids"]) - len(word_ids) + 1)
            )
        ]

        return pd.DataFrame(filtered_examples), word_ids

    def extract_embeddings(self, word, word_label):
        """
        Extract embeddings for a given word and return a DataFrame.
        """
        filtered_df, word_ids = self.filter_dataset(word)

        if filtered_df.empty:
            print(f"No occurrences of '{word}' found.")
            return None

        # Store embeddings
        embeddings_dict = {}

        for index, row in filtered_df.iterrows():
            input_ids = row["input_ids"]
            
            # Find occurrences of word in input_ids
            word_indices = [
                i for i in range(len(input_ids) - len(word_ids) + 1)
                if input_ids[i:i+len(word_ids)] == word_ids
            ]

            # Tokenize input
            text = self.tokenizer.decode(input_ids, skip_special_tokens=True)
            inputs = self.tokenizer(text, return_tensors="pt", truncation=True, max_length=512)

            # Get last hidden states
            with torch.no_grad():
                outputs = self.model(**inputs, output_hidden_states=True)

            last_hidden_state = outputs.hidden_states[-1]

            row_embeddings = []
            for idx in word_indices:
                word_span = list(range(idx, idx + len(word_ids)))
                word_embedding = last_hidden_state[:, word_span, :].mean(dim=1)
                row_embeddings.append(word_embedding.squeeze().numpy())

            embeddings_dict[index] = row_embeddings

        # Convert to DataFrame
        embeddings_df = pd.DataFrame(list(embeddings_dict.items()), columns=["row_index", "embeddings"])
        embeddings_df["label"] = word_label  # Assign the provided label

        return embeddings_df

    def process_multiple_words(self, words_labels):
        """
        Process multiple words and combine embeddings.
        words_labels: List of tuples [(word1, label1), (word2, label2), ...]
        """
        all_embeddings = []
        labels = []

        for word, label in words_labels:
            print(f"Processing '{word}' with label {label}...")
            word_df = self.extract_embeddings(word, label)
            if word_df is not None:
                for _, row in word_df.iterrows():
                    for embedding in row["embeddings"]:
                        all_embeddings.append(embedding)
                        labels.append(label)

        # Convert to numpy arrays
        all_embeddings = np.array(all_embeddings)
        labels = np.array(labels)

        # Save dataset
        np.save("embeddings.npy", all_embeddings)
        np.save("labels.npy", labels)

        print(f"Saved {all_embeddings.shape[0]} embeddings.")
        return all_embeddings, labels

In [11]:
# Define model and dataset paths
model_dir = "/Users/jonasklein/biasintransformers/downloaded_model"
dataset_path = "/Users/jonasklein/biasintransformers/downloaded_dataset/data-00000-of-00001.arrow"

# List of words and their labels
words_labels = [
    ("man", 1),
    ("vrouw", 0),
    ("broer", 1),
    ("zus", 0),
    ("zoon", 1),
    ("dochter", 0),
    ("neef", 1),
    ("nicht", 0),
    ("vader", 1),
    ("moeder", 0),
    ("opa", 1),
    ("oma", 0),
    ("kleinzoon", 1),
    ("kleindochter", 0),
    ("grootvader", 1),
    ("grootmoeder", 0),
    ("oom", 1),
    ("tante", 0),
    ("papa", 1),
    ("mama", 0),
    ("jongen", 1),
    ("meisje", 0),
    ("jongetje", 1),
    ("meid", 0),
    ("schoonvader", 1),
    ("schoonmoeder", 0),
    ("schoonzoon", 1),
    ("schoondochter", 0),
    ("stiefvader", 1),
    ("stiefmoeder", 0),
    ("stiefzoon", 1),
    ("stiefdochter", 0),
    ("peetvader", 1),
    ("peetmoeder", 0),
    ("bruidegom", 1),
    ("bruid", 0),
    ("meneer", 1),
    ("mevrouw", 0),
    ("mijnheer", 1),
    ("heer", 1),
    ("dame", 0),
    ("kerel", 1),
    ("mister", 1),
    ("miss", 0),
    ("mr", 1),
    ("ms", 0),
    ("prins", 1),
    ("prinses", 0),
    ("koning", 1),
    ("koningin", 0),
    ("lord", 1),
    ("lady", 0),
    ("baron", 1),
    ("barones", 0),
    ("hertog", 1),
    ("hertogin", 0),
    ("monnik", 1),
    ("non", 0),
    ("hijzelf", 1),
    ("zijzelf", 0),
    ("mannelijk", 1),
    ("vrouwelijk", 0),
    ("gentleman", 1),
    ("gozer", 1),
    ("wijf", 0)
]


# Initialize extractor
extractor = WordEmbeddingExtractor(model_dir, dataset_path)

# Process multiple words
embeddings, labels = extractor.process_multiple_words(words_labels)

# Verify shapes
print("Final embeddings shape:", embeddings.shape)
print("Final labels shape:", labels.shape)


Some weights of BertModel were not initialized from the model checkpoint at /Users/jonasklein/biasintransformers/downloaded_model and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded dataset with 19968 entries.
Processing 'man' with label 1...
Processing 'vrouw' with label 0...
Processing 'broer' with label 1...
Processing 'zus' with label 0...
Processing 'zoon' with label 1...
Processing 'dochter' with label 0...
Processing 'neef' with label 1...
Processing 'nicht' with label 0...
Processing 'vader' with label 1...
Processing 'moeder' with label 0...
Processing 'opa' with label 1...
Processing 'oma' with label 0...
Processing 'kleinzoon' with label 1...
Processing 'kleindochter' with label 0...
Processing 'grootvader' with label 1...
Processing 'grootmoeder' with label 0...
Processing 'oom' with label 1...
Processing 'tante' with label 0...
Processing 'papa' with label 1...
Processing 'mama' with label 0...
Processing 'jongen' with label 1...
Processing 'meisje' with label 0...
Processing 'jongetje' with label 1...
Processing 'meid' with label 0...
Processing 'schoonvader' with label 1...
Processing 'schoonmoeder' with label 0...
Processing 'schoonzoon' wit